# Data Exploration

**Quito Public Transport Intelligence Platform** — notebook 01 of the pipeline.

Establishes what the three raw datasets contain, which time window is trustworthy,
and which measures can be used downstream.

Every section follows the same shape: a line on *why* the check is run, the code, then a
**Finding** block recording the result. The findings are the deliverable — this notebook
is the data-quality record for the project, not a scratchpad.

---

## 0. Configuration

Two windows, deliberately different. `RAW_*` is the span of the **traffic** collection;
`LOCAL_*` is the subset that survives the quality checks in sections 5–7.

The notebook runs unchanged on Databricks (where `spark` and `.display()` already exist)
and on plain PySpark (`QUITO_RAW_ROOT=/path/to/s3_raw`), so the findings can be re-verified
anywhere the raw zone is mounted.

In [2]:
import os
from pyspark.sql import functions as F, DataFrame

# Raw zone root. Databricks volume by default; override with QUITO_RAW_ROOT elsewhere.
BASE = os.environ.get(
    "QUITO_RAW_ROOT",
    "/Volumes/workspace/default/data-engineering/quito-transport-platform/s3_raw",
)

# Outside Databricks there is no ambient `spark` and DataFrame has no .display():
# create a local session and make .display() an alias of .show().
try:
    spark  # noqa: F821  (Databricks injects it)
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("01_exploring").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
if not hasattr(DataFrame, "display"):
    DataFrame.display = lambda self, n=60: self.show(n, truncate=False)

# Span of the traffic collection. network and weather keep accumulating daily
# snapshots beyond this (to 2026-10-09 at the time of writing); section 10 uses those.
RAW_START, RAW_END = "2026-08-20", "2026-09-02"

# Analysis window: complete LOCAL days with hourly traffic coverage (derived in section 6)
LOCAL_START, LOCAL_END = "2026-08-25", "2026-08-31"

UTC_OFFSET_HOURS = 5   # Quito is UTC-5 year-round; Ecuador has no DST

---

## 1. Physical layout

Walk the tree before reading anything. Partition keys, nesting depth and file counts are all
visible from the filesystem and cost nothing to inspect.

In [4]:
for root, dirs, files in os.walk(BASE):
    depth = root[len(BASE):].count(os.sep)
    if depth <= 3:
        print(f"{'  ' * depth}{os.path.basename(root)}/  ({len(files)} files)")

s3_raw/  (0 files)
  network/  (0 files)
    year=2026/  (0 files)
      month=09/  (0 files)
      month=08/  (0 files)
      month=10/  (0 files)
  weather/  (0 files)
    year=2026/  (0 files)
      month=09/  (0 files)
      month=08/  (0 files)
      month=06/  (0 files)
      month=10/  (0 files)
  traffic/  (0 files)
    year=2026/  (0 files)
      month=09/  (0 files)
      month=08/  (0 files)


`traffic` reports 0 files at the day level because it has a fourth partition level the others lack:

In [6]:
print(sorted(os.listdir(f"{BASE}/traffic/year=2026/month=08/day=20")))

['hour=00', 'hour=01', 'hour=06', 'hour=12', 'hour=18']


> **Finding**
>
> This is not one partitioned dataset but three independent ones.
>
> - `network` and `weather` are partitioned `year / month / day`
> - `traffic` adds `hour`
>
> Pointing Spark at `BASE` raises `CONFLICTING_DIRECTORY_STRUCTURES`. Each dataset must be read separately.

---

## 2. Schemas

Parquet keeps its schema in the file footer, so this reads one footer per dataset — no data scanned.

`network` needs care. Each daily partition holds **three files with three different schemas**
(`routes`, `stops`, `route_stops`). Pointing Spark at the folder does not merge them: it takes the
schema of whichever file it opens first and reads every other file through it, silently producing
garbage rows (an earlier version of this notebook reported "27 435 network rows" that way). Each
table is read by file name instead. The glob drops the `year=/month=/day=` directories from the
path Spark sees, so `basePath` is set to recover them as partition columns.

In [9]:
traffic_raw = spark.read.parquet(f"{BASE}/traffic")
weather_raw = spark.read.parquet(f"{BASE}/weather")

def read_network_table(name):
    return (spark.read
                 .option("basePath", f"{BASE}/network")
                 .parquet(f"{BASE}/network/year=*/month=*/day=*/{name}.parquet"))

routes_raw      = read_network_table("routes")
stops_raw       = read_network_table("stops")
route_stops_raw = read_network_table("route_stops")

for name, df in [("traffic", traffic_raw), ("weather", weather_raw),
                 ("routes", routes_raw), ("stops", stops_raw), ("route_stops", route_stops_raw)]:
    print(f"=== {name} ===")
    df.printSchema()

=== traffic ===
root
 |-- route_id: string (nullable = true)
 |-- route_ref: string (nullable = true)
 |-- route_name: string (nullable = true)
 |-- sampled_at: string (nullable = true)
 |-- local_hour: long (nullable = true)
 |-- local_weekday: long (nullable = true)
 |-- waypoint_count: long (nullable = true)
 |-- mean_hop_km: double (nullable = true)
 |-- distance_meters: double (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- mean_speed_kmh: double (nullable = true)
 |-- segments_total: long (nullable = true)
 |-- segments_known: long (nullable = true)
 |-- congestion_known_pct: double (nullable = true)
 |-- congestion_low: long (nullable = true)
 |-- congestion_moderate: long (nullable = true)
 |-- congestion_heavy: long (nullable = true)
 |-- congestion_severe: long (nullable = true)
 |-- congestion_unknown: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer

> **Finding** — five tables at three grains, not three peer datasets
>
> | dataset | grain | role |
> |---|---|---|
> | `traffic` | route x hour | fact |
> | `weather` | time only, no location column | time dimension |
> | `routes`, `stops` | one row per entity, re-fetched daily | slowly-changing dimensions |
> | `route_stops` | route x stop, re-fetched daily | bridge between the two |
>
> Two consequences for the model:
>
> - Weather is a **single observation point for the whole city**, so the join is time-only.
>   Nothing can be said about weather varying by route.
> - The network tables are **daily snapshots**. Joining them to the fact on date would fan it out
>   by the number of days; the dimensions must be collapsed to one row per entity first (section 10).

---

## 3. Raw volumes

Filtering on the partition columns lets Spark prune directories instead of scanning everything.

In [12]:
def scan_window(df):
    return df.filter(F.make_date("year", "month", "day").between(RAW_START, RAW_END))

traffic = scan_window(traffic_raw)
weather = scan_window(weather_raw)
routes, stops, route_stops = (scan_window(df) for df in (routes_raw, stops_raw, route_stops_raw))

for name, df in [("traffic", traffic), ("weather", weather),
                 ("routes", routes), ("stops", stops), ("route_stops", route_stops)]:
    print(f"{name:12s} {df.count():6d}")

traffic        3108
weather         574
routes         7205
stops          6790
route_stops   13440


> **Finding**
>
> | dataset | rows | per day |
> |---|---|---|
> | `traffic` | 3 108 | varies — see section 6 |
> | `weather` | 574 | 41 |
> | `routes` | 7 205 | 514–515 |
> | `stops` | 6 790 | 485 |
> | `route_stops` | 13 440 | 960 |
>
> 574 / 14 days = **exactly 41 rows per day**. That is not hourly observation data.
> 41 = 1 + 40, and 40 is what OpenWeather's 5-day / 3-hour forecast endpoint returns.
> Tested in section 4.
>
> The network tables are ~14 near-identical copies of the same snapshot. Only 119 of 515 routes
> have any stop mapped (960 rows / ~8 stops per route), as the README's limitations table states.

---

## 4. Weather composition

`record_type` exists for a reason, and the usual reason is that observations and forecasts share the file.

In [15]:
weather.groupBy("record_type") \
       .count() \
       .display()

+-----------+-----+
|record_type|count|
+-----------+-----+
|forecast   |560  |
|current    |14   |
+-----------+-----+



`timestamp` and `fetched_at` arrive as strings, so they sort as text rather than as time. Parse both:

- `ts` — when the weather happened
- `fetched` — when the API was called

A large gap between the two is the signature of a forecast.

In [17]:
w = weather \
    .withColumn("ts", F.to_timestamp("timestamp")) \
    .withColumn("fetched", F.to_timestamp("fetched_at"))

w.filter(F.make_date("year", "month", "day") == F.lit("2026-08-20")) \
 .select("record_type", "fetched", "ts", "temp_c") \
 .orderBy("ts") \
 .display()

+-----------+--------------------------+-------------------+------+
|record_type|fetched                   |ts                 |temp_c|
+-----------+--------------------------+-------------------+------+
|current    |2026-08-20 05:00:04.982198|2026-08-20 05:00:05|10.33 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 06:00:00|10.33 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 09:00:00|10.29 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 12:00:00|10.21 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 15:00:00|17.07 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 18:00:00|17.77 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-20 21:00:00|15.53 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-21 00:00:00|11.88 |
|forecast   |2026-08-20 05:00:04.982198|2026-08-21 03:00:00|9.49  |
|forecast   |2026-08-20 05:00:04.982198|2026-08-21 06:00:00|7.83  |
|forecast   |2026-08-20 05:00:04.982198|2026-08-21 09:00:00|6.87  |
|forecast   |2026-08-20 05:00:04.982198|2026-08-

> **Finding**
>
> 14 `current` + 560 `forecast`, and 560 = 14 x 40. One API call per day at 05:00 UTC (midnight local)
> wrote one observation plus a 5-day forecast at 3-hour steps.
>
> **Timestamps are UTC.** The `+00:00` suffix says so, and the temperature curve agrees —
> 10.3 C at 05:00 UTC and 17.8 C at 18:00 UTC are midnight and 1 pm local, the right shape for Quito.
>
> **Only 14 real observations exist**, one per day, all at midnight local. Not enough for an hourly join.
>
> The forecasts rescue this. Each target time is forecast several times at different lead times, so
> keeping the shortest-lead forecast per target yields a usable 3-hourly series — with lead time
> retained as a column so the provenance stays honest.

---

## 5. Traffic time base

Three time representations exist in `traffic`: the `hour` partition, the `local_hour` column, and the
`sampled_at` string. Weather is UTC; if `hour` turned out to be local, every joined reading would be
five hours out of place.

In [20]:
t = traffic \
    .withColumn("ts", F.to_timestamp("sampled_at"))

t.groupBy("hour", "local_hour") \
 .count() \
 .orderBy("hour") \
 .display()

+----+----------+-----+
|hour|local_hour|count|
+----+----------+-----+
|0   |19        |196  |
|1   |20        |140  |
|2   |21        |112  |
|3   |22        |112  |
|4   |23        |112  |
|5   |0         |112  |
|6   |1         |182  |
|7   |2         |112  |
|8   |3         |112  |
|9   |4         |112  |
|10  |5         |112  |
|11  |6         |112  |
|12  |7         |182  |
|13  |8         |112  |
|14  |9         |112  |
|15  |10        |112  |
|16  |11        |126  |
|17  |12        |126  |
|18  |13        |182  |
|19  |14        |126  |
|20  |15        |126  |
|21  |16        |126  |
|22  |17        |126  |
|23  |18        |126  |
+----+----------+-----+



> **Finding**
>
> `local_hour = hour - 5` for every row, with no hour splitting across two local hours.
> So **`hour` is UTC** and `local_hour` is Quito time. No DST ambiguity.
>
> This matters for the join: the weather forecast grid sits at 00, 03, 06 … 21 UTC and traffic's
> partition is UTC too, so those hours align exactly with no conversion.

---

## 6. Coverage grid

`pivot` turns each distinct hour into its own column, producing a day-by-hour heatmap of row counts.
Nulls are hours never sampled; uneven numbers within a row mean the route set changed mid-day.

In [23]:
t.withColumn("date", F.make_date("year", "month", "day")) \
 .groupBy("date") \
 .pivot("hour") \
 .count() \
 .orderBy("date") \
 .display()

+----------+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+
|date      |0  |1   |2   |3   |4   |5   |6   |7   |8   |9   |10  |11  |12  |13  |14  |15  |16  |17  |18  |19  |20  |21  |22  |23  |
+----------+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+
|2026-08-20|14 |14  |NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|
|2026-08-21|14 |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|
|2026-08-22|14 |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|
|2026-08-23|14 |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|
|2026-08-24|14 |NULL|NULL|NULL|NULL|NULL|14  |NULL|NULL|NULL|NULL|NULL|14  |

> **Finding**
>
> Every non-null cell is exactly **14** — fourteen routes, no exceptions. But the collection cadence
> changed mid-run:
>
> | period (UTC) | cadence |
> |---|---|
> | Aug 20 – Aug 24 12:00 | 4 samples/day, hours 0 / 6 / 12 / 18 |
> | Aug 24 16:00 – Sep 01 23:00 | hourly, complete |
> | Sep 02 | truncated at 01:00 |
>
> The arithmetic reconciles: 70 + 168 + 154 + 2688 + 28 = 3 108, the exact row count.
> Nothing missing, nothing duplicated.
>
> Mixing the sparse regime with the hourly one would bias every hourly average toward hours 0/6/12/18,
> so the sparse days are dropped. The research questions are local-time questions — rush hour,
> weekday effects — so the window is cut to complete **local** days:
>
> **2026-08-25 to 2026-08-31. Seven days, exactly one calendar week.**

---

## 7. Lock the analysis window

Build a real timestamp from the partition columns, shift it to local time, filter on the local date.

Deriving the window rather than hardcoding date conditions has a second payoff: it produces
`local_date`, which every daily aggregation downstream reuses.

In [26]:
utc_ts = F.expr("make_timestamp(year, month, day, hour, 0, 0)")

analysis = t \
    .withColumn("utc_ts", utc_ts) \
    .withColumn("local_ts", utc_ts - F.expr(f"INTERVAL {UTC_OFFSET_HOURS} HOURS")) \
    .withColumn("local_date", F.to_date("local_ts")) \
    .filter(F.col("local_date").between(LOCAL_START, LOCAL_END))

print("rows:", analysis.count(), "(expected 2352 = 7 days x 24 h x 14 routes)")

rows: 2352 (expected 2352 = 7 days x 24 h x 14 routes)


Cross-check the derived local time against the `local_hour` column the ingestion pipeline wrote.
Two independent derivations agreeing is worth more than either alone.

In [28]:
print("timezone mismatches:", analysis.filter(F.hour("local_ts") != F.col("local_hour")).count())

timezone mismatches: 0


In [29]:
analysis.groupBy("route_id", "route_ref", "route_name") \
        .count() \
        .orderBy("route_ref") \
        .display()

+--------+--------------------------------+---------------------------------------+-----+
|route_id|route_ref                       |route_name                             |count|
+--------+--------------------------------+---------------------------------------+-----+
|12113850|C1                              |Trolebus C1 El Labrador => El Recreo   |168  |
|12113848|C1                              |Trolebus C1 El Recreo => El Labrador   |168  |
|85965   |C4                              |Trolebus C4 Quitumbe => La Colón       |168  |
|2006760 |C4                              |Trolebus C4 La Colón => Quitumbe       |168  |
|85972   |C6                              |Trolebus C6 Quitumbe - El Recreo       |168  |
|12113852|C6                              |Trolebus C6 El Recreo => Quitumbe      |168  |
|12110365|E1                              |Ecovía E1 T. Sur - Las Universidades   |168  |
|12110382|E1R                             |Ecovía E1R T. Sur - Terminal El Recreo |168  |
|12113856|

> **Finding**
>
> 2 352 rows · 0 timezone mismatches · 14 routes at 168 samples each (7 x 24).
>
> The fact table is a perfect rectangle. No route dropped out mid-week.

---

## 8. Nulls and ranges

`isNull()` returns true/false per row; casting to int and summing turns that into a null count per
column. Which columns fail *together* usually reveals why.

In [32]:
MEASURES = ["waypoint_count", "mean_hop_km", "distance_meters", "duration_seconds",
            "mean_speed_kmh", "segments_total", "segments_known", "congestion_known_pct",
            "congestion_low", "congestion_moderate", "congestion_heavy",
            "congestion_severe", "congestion_unknown"]

analysis.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in MEASURES]) \
        .display()

+--------------+-----------+---------------+----------------+--------------+--------------+--------------+--------------------+--------------+-------------------+----------------+-----------------+------------------+
|waypoint_count|mean_hop_km|distance_meters|duration_seconds|mean_speed_kmh|segments_total|segments_known|congestion_known_pct|congestion_low|congestion_moderate|congestion_heavy|congestion_severe|congestion_unknown|
+--------------+-----------+---------------+----------------+--------------+--------------+--------------+--------------------+--------------+-------------------+----------------+-----------------+------------------+
|0             |0          |0              |0               |0             |0             |0             |0                   |0             |0                  |0               |0                |0                 |
+--------------+-----------+---------------+----------------+--------------+--------------+--------------+--------------------+-----

Percentiles expose the impossible values routing APIs commonly emit — zero or 300 km/h speeds,
zero-second durations, negative distances, percentages above 100 — and show whether a long tail
would distort any mean.

In [34]:
analysis.select("mean_speed_kmh", "duration_seconds", "distance_meters", "congestion_known_pct") \
        .summary("min", "25%", "50%", "75%", "max") \
        .display()

+-------+--------------+----------------+---------------+--------------------+
|summary|mean_speed_kmh|duration_seconds|distance_meters|congestion_known_pct|
+-------+--------------+----------------+---------------+--------------------+
|min    |18.03         |855.2           |8460.8         |0.0                 |
|25%    |21.49         |1955.2          |13804.2        |12.1                |
|50%    |23.49         |2778.9          |19057.6        |37.4                |
|75%    |30.55         |3506.7          |25796.0        |56.4                |
|max    |39.74         |5341.0          |31036.9        |95.3                |
+-------+--------------+----------------+---------------+--------------------+



> **Finding**
>
> Zero nulls across all thirteen measures.
>
> | measure | min | median | max | verdict |
> |---|---|---|---|---|
> | `mean_speed_kmh` | 18.0 | 23.5 | 39.7 | plausible |
> | `duration_seconds` | 855 | 2 779 | 5 341 | 14–89 min, plausible |
> | `distance_meters` | 8 461 | 19 058 | 31 037 | 8.5–31 km, plausible |
> | `congestion_known_pct` | 0.0 | 37.4 | 95.3 | **weak** |
>
> `mean_speed_kmh` derives from distance and duration, which the routing API returns regardless of
> traffic-layer coverage. Speed is trustworthy.
>
> `congestion_known_pct` is not. In half the samples fewer than 37 % of road segments reported
> congestion data; the bottom quartile is under 12 %; some samples have none at all. Comparing
> `congestion_heavy` between a row at 80 % coverage and one at 10 % compares different things.

---

## 9. Is the missingness structured?

Random gaps are a precision problem. Gaps that correlate with route or time of day are a **bias**
problem. This decides whether congestion is usable at all.

In [37]:
analysis.groupBy("route_id", "route_ref") \
        .agg(F.round(F.min("distance_meters"), 0).alias("min_dist"), \
             F.round(F.max("distance_meters"), 0).alias("max_dist"), \
             F.round(100 * (F.max("distance_meters") - F.min("distance_meters")) \
                     / F.min("distance_meters"), 1).alias("spread_pct"), \
             F.round(F.avg("congestion_known_pct"), 1).alias("avg_known")) \
        .orderBy(F.desc("spread_pct")) \
        .display()

+--------+--------------------------------+--------+--------+----------+---------+
|route_id|route_ref                       |min_dist|max_dist|spread_pct|avg_known|
+--------+--------------------------------+--------+--------+----------+---------+
|85972   |C6                              |17391.0 |20901.0 |20.2      |28.8     |
|12113854|E6                              |10375.0 |11869.0 |14.4      |25.0     |
|12113852|C6                              |21419.0 |24301.0 |13.5      |31.7     |
|85965   |C4                              |26420.0 |29552.0 |11.9      |39.2     |
|2006760 |C4                              |28021.0 |31037.0 |10.8      |33.9     |
|12113848|C1                              |20382.0 |21983.0 |7.9       |47.3     |
|2021069 |Metro-Bus Corredor Central Norte|26142.0 |28218.0 |7.9       |43.0     |
|12113850|C1                              |18984.0 |20091.0 |5.8       |49.4     |
|85969   |Metrobus-Q                      |25506.0 |26928.0 |5.6       |41.7     |
|121

Two questions in one table.

- **Coverage** — if poor coverage concentrates in a few routes, drop those routes rather than the
  whole column.
- **Geometry stability** — a bus route's path is fixed, so min and max distance should be nearly
  identical. A spread beyond a percent or two means the routing API returned different paths on
  different calls, and speed is not comparable across time for that route.

In [39]:
analysis.groupBy("local_hour") \
        .agg(F.round(F.avg("congestion_known_pct"), 1).alias("avg_known"),
             F.round(F.avg("mean_speed_kmh"), 1).alias("avg_speed")) \
        .orderBy("local_hour") \
        .display()

+----------+---------+---------+
|local_hour|avg_known|avg_speed|
+----------+---------+---------+
|0         |20.6     |33.5     |
|1         |11.2     |34.0     |
|2         |6.8      |34.2     |
|3         |7.9      |33.8     |
|4         |1.8      |33.1     |
|5         |4.2      |28.8     |
|6         |9.4      |23.4     |
|7         |23.6     |21.4     |
|8         |31.9     |21.4     |
|9         |38.1     |22.6     |
|10        |42.4     |21.8     |
|11        |43.3     |21.6     |
|12        |48.5     |21.3     |
|13        |53.2     |21.6     |
|14        |55.2     |22.3     |
|15        |55.9     |22.1     |
|16        |49.7     |21.8     |
|17        |52.8     |21.4     |
|18        |51.0     |21.7     |
|19        |56.0     |23.0     |
|20        |56.3     |24.7     |
|21        |56.5     |26.8     |
|22        |46.5     |29.1     |
|23        |27.7     |31.7     |
+----------+---------+---------+



> **Finding**
>
> **Geometry is not stable.** Six corridors vary by more than 10 % in routed distance across the
> week (C6 by 20 %, E6 and C4 by 11–14 %). Mapbox is choosing different paths between the same
> waypoints at different times, so `mean_speed_kmh` on those routes partly reflects *which road*,
> not only *how congested*. `duration_seconds` is the honest measure of "how long the corridor
> takes"; speed comparisons across hours need the distance held in view. E1R (0.0 %) and E1 (2.4 %)
> are the stable reference corridors.
>
> **Congestion missingness is structured by hour.** Coverage (`avg_known`) climbs from 2 % at
> 04:00 to 56 % at 19:00–21:00 and falls back overnight. The gaps concentrate in exactly the hours
> under study, so any hourly `congestion_*` average is biased. Coverage by route is flatter
> (25–49 %), so the bias is temporal, not spatial.
>
> **Speed tells the story on its own.** Average speed is 33–34 km/h from midnight to 04:00, drops
> to 21–22 km/h for the whole 07:00–18:00 span, and recovers through the evening. That is the
> rush-hour signal the project was built to find, with no dependence on the congestion layer.
>
> Decision: `duration_seconds` and `mean_speed_kmh` are the primary measures; `congestion_*` stays
> in the fact table as supporting detail with `congestion_known_pct` alongside so consumers can
> filter on coverage.

---

## 10. Open questions: the network across snapshots

`routes`, `stops` and `route_stops` are re-fetched daily. Two things decide how they are modelled:
whether the snapshots actually differ, and whether the traffic routes can be joined to them.
The full span of snapshots is used here, not just the traffic window.

In [42]:
from pyspark.sql import Window as W

def with_snapshot_date(df):
    return df.withColumn("snapshot_date", F.make_date("year", "month", "day"))

snap = (with_snapshot_date(routes_raw).groupBy("snapshot_date").agg(F.count("*").alias("routes"))
        .join(with_snapshot_date(stops_raw).groupBy("snapshot_date").agg(F.count("*").alias("stops")),
              "snapshot_date")
        .join(with_snapshot_date(route_stops_raw).groupBy("snapshot_date").agg(F.count("*").alias("route_stops")),
              "snapshot_date"))

prev = W.orderBy("snapshot_date")
changes = (snap
           .withColumn("d_routes", F.col("routes") - F.lag("routes").over(prev))
           .withColumn("d_stops", F.col("stops") - F.lag("stops").over(prev))
           .withColumn("d_route_stops", F.col("route_stops") - F.lag("route_stops").over(prev))
           .orderBy("snapshot_date"))

print("snapshots:", snap.count(),
      "| distinct route_id overall:", routes_raw.select("route_id").distinct().count(),
      "| distinct stop_id overall:", stops_raw.select("stop_id").distinct().count())
changes.filter("d_routes <> 0 or d_stops <> 0 or d_route_stops <> 0").display()

snapshots: 52 | distinct route_id overall: 515 | distinct stop_id overall: 486
+-------------+------+-----+-----------+--------+-------+-------------+
|snapshot_date|routes|stops|route_stops|d_routes|d_stops|d_route_stops|
+-------------+------+-----+-----------+--------+-------+-------------+
|2026-08-29   |514   |485  |960        |-1      |0      |0            |
|2026-09-14   |515   |486  |961        |1       |1      |1            |
|2026-09-15   |514   |485  |960        |-1      |-1     |-1           |
|2026-09-19   |515   |486  |961        |1       |1      |1            |
|2026-09-21   |514   |485  |960        |-1      |-1     |-1           |
|2026-09-22   |515   |486  |961        |1       |1      |1            |
|2026-09-24   |515   |485  |960        |0       |-1     |-1           |
|2026-09-25   |513   |485  |960        |-2      |0      |0            |
|2026-09-27   |515   |486  |961        |2       |1      |1            |
|2026-09-28   |513   |485  |960        |-2      |-1     |

Whether a route that disappears is *withdrawn* or merely *missing from one response* shows in
how it comes back. Count, per route, the snapshots it is absent from.

In [44]:
all_dates = routes_raw.select(F.make_date("year", "month", "day").alias("snapshot_date")).distinct()
all_routes = routes_raw.select("route_id").distinct()
n_snapshots = all_dates.count()

presence = (all_routes.crossJoin(all_dates)
            .join(with_snapshot_date(routes_raw).select("route_id", "snapshot_date").distinct(),
                  ["route_id", "snapshot_date"], "left_semi")
            .groupBy("route_id").agg(F.count("*").alias("present")))

(presence.filter(F.col("present") < n_snapshots)
         .join(routes_raw.select("route_id", "route_ref", "route_name").distinct(), "route_id")
         .withColumn("absent", F.lit(n_snapshots) - F.col("present"))
         .orderBy("absent", ascending=False)
         .display())

+--------+-------+-------------+----------------------------+------+
|route_id|present|route_ref    |route_name                  |absent|
+--------+-------+-------------+----------------------------+------+
|6433119 |20     |7 DE MAYO-144|Marin - Rocio de Guamani    |32    |
|6427512 |41     |TROLEBUS-Q11 |Estacion El Recreo - Solanda|11    |
|6427512 |41     |TROLEBUS-Q11 |Estación El Recreo - Solanda|11    |
+--------+-------+-------------+----------------------------+------+



> **Finding**
>
> 52 snapshots, 2026-08-19 to 2026-10-09. The counts flip **515 → 513 → 515 → 514** and back,
> but only **two routes are ever involved**:
>
> | route | present | pattern |
> |---|---|---|
> | `6433119` *7 DE MAYO-144* | 20 / 52 | continuous to 08-28, then scattered single days |
> | `6427512` *TROLEBUS-Q11* | 41 / 52 | continuous to 09-24, then scattered; name edited *Estacion → Estación* mid-way |
>
> Neither is a withdrawal: both keep reappearing, and one was being edited at the time. This is
> **flapping** — a relation hovering on the edge of what the Overpass query returns (a mapper's
> work in progress, or a bbox edge case), not a change in Quito's network. The `routes` quality
> suite's minimum-row-count expectation is there for gross truncation; a two-route wobble passes it.
>
> Consequences for the dimension tables:
>
> - **Do not build `dim_route` from the latest snapshot alone.** On a day either route is absent,
>   it would vanish from the dimension and its fact rows would lose their context. Build the
>   dimensions from the **union of all snapshots**, keeping the most recently seen version of each
>   entity plus `first_seen_date` / `last_seen_date` / `snapshots_present`.
> - "Removed from the network" must mean *absent for N consecutive snapshots*, not *absent today*.
> - No `sequence_source` or `snap_distance_m` column exists in any snapshot yet. The geometry-based
>   stop ordering (`network_loader`, 2026-10) has not produced data — the deployed Lambda predates
>   it. Downstream code must treat those columns as optional.

Do `traffic.route_id` and `routes.route_id` overlap? And which trunk relations has the traffic
sampler never covered?

In [47]:
TRUNK_PATTERN = "(?i)Trole|Ecov|MetroBus|Metrobus"   # mirrors ingestion.constants.TRUNK_ROUTE_PATTERN

sampled = traffic_raw.select("route_id").distinct()
trunk = (routes_raw.filter(F.col("route_name").rlike(TRUNK_PATTERN))
                   .select("route_id", "route_ref", "route_name").distinct())

print("traffic routes:", sampled.count(),
      "| of which in routes:", sampled.join(all_routes, "route_id").count())
print("trunk relations in OSM:", trunk.count(),
      "| sampled:", trunk.join(sampled, "route_id").count())
trunk.join(sampled, "route_id", "left_anti").orderBy("route_ref").display()

traffic routes: 14 | of which in routes: 14
trunk relations in OSM: 20 | sampled: 14
+--------+------------------+---------------------------------------+
|route_id|route_ref         |route_name                             |
+--------+------------------+---------------------------------------+
|12110323|E1                |Ecovía E1 Las Universidades - T. Sur   |
|12110381|E1R               |Ecovía E1R Terminal El Recreo - T. Sur |
|2025942 |E3                |Ecovía E3 Río Coca - Playón de la Marín|
|12110437|E3                |Ecovia E3 Playón de la Marín - Río Coca|
|2024092 |Ruta de Integració|Integració Trolebus                    |
|85976   |Ruta de Integració|Integració Ecovía                      |
+--------+------------------+---------------------------------------+



> **Finding**
>
> All 14 sampled routes exist in `routes` — the fact-to-dimension join is clean.
>
> 20 OSM relations match the trunk pattern. Six were never sampled: the two *Ruta de Integración*
> relations carry no stop members (nothing to route through), and the four Ecovía relations
> **E1, E1R, E3 (both directions)** are the ones `build_corridors` rejected because OSM lists their
> stops out of geographic order (`CORRIDOR_MAX_MEAN_HOP_KM`). The geometry-based ordering that
> recovers them is in the code but not yet in the data, so the series stays at 14 corridors for
> this window. When it lands, traffic gains four corridors and two columns (`sequence_source`,
> `stops_excluded`); the fact table must accept both the old and the new shape.

---

## Summary

### Established

- Five raw tables at three grains; `network` is three tables that must be read by file name.
- All timestamps are UTC. Quito is UTC-5, no DST. `traffic.local_hour` verified correct.
- Analysis window: **2026-08-25 to 2026-08-31 local** — seven days, one full calendar week.
- Fact table: **2 352 rows = 14 routes x 24 hours x 7 days.** Complete, no nulls, no duplicates.
- `duration_seconds` is the primary measure; `mean_speed_kmh` is trustworthy but reflects path
  choice on six corridors; `congestion_*` is supporting detail only.
- Speed alone shows the rush-hour signal: 33–34 km/h overnight, 21–22 km/h from 07:00 to 18:00.
- Network snapshots differ only by Overpass truncation; dimensions come from the union of all
  snapshots, not the latest one.
- All 14 traffic routes join to `routes`. Four Ecovía trunk corridors are not yet sampled.

### Known limitations

Carry these into the README rather than leaving them to be discovered.

- Weather has no spatial dimension — one observation point for the whole city.
- Only one real weather observation per day; the hourly series is reconstructed from shortest-lead forecasts.
- `congestion_*` coverage is structured by hour (2 % at 04:00, 56 % at 20:00), so hourly congestion averages are biased.
- Routed distance varies up to 20 % on the same corridor across the week.
- 2 352 rows is small. Enough for descriptive analysis and a clean pipeline; not enough for modelling.
- `sequence_source` / `snap_distance_m` (geometry-ordered stops) are absent from every snapshot to date.